# Оригинал или перевод: выявление переведённых текстов на русском языке как задача бинарной классификации на материале платформы Ficbook.group

## Установка пакетов

In [ ]:
!pip install --upgrade transformers datasets accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: Operat

### Загрузка текстов

У нас уже есть тексты, сохранённые в файлах `origs_NER.rar` и `trans_NER.rar`.

Для корректного запуска тетрадки нужно выгрузить их в корневую папку колаба и продолжить запускать ячейки.

In [10]:
!apt-get install unrar
!unrar x origs_NER.rar
!unrar x trans_NER.rar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from origs_NER.rar


Would you like to replace the existing file content/origs_NER/130403.txt
 33278 bytes, modified on 2026-01-27 11:37
with a new one
 33278 bytes, modified on 2026-01-27 11:37

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit Q

Program aborted

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from trans_NER.rar


Would you like to replace the existing file content/trans_NER/299072.txt
 16372 bytes, modified on 2026-01-27 12:59
with a new one
 16372 bytes, modified on 2026-01-27 12:59

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit Q

Program aborted


In [11]:
!ls /content/content/origs_NER | wc -l

855


In [12]:
!ls /content/content/trans_NER | wc -l

855


## Импорты

In [13]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from tqdm import tqdm
import torch
from transformers import Trainer, TrainingArguments

In [14]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import torch

In [15]:
def load_texts(folder, label):
    texts = []
    labels = []

    for file in os.listdir(folder):
        if file.endswith(".txt"):
            with open(os.path.join(folder, file), encoding="utf-8") as f:
                text = f.read()
                # убираю ссылку на фанфик
                text = text.split('\n', 1)[-1]

                texts.append(text)
                labels.append(label)

    return texts, labels

In [16]:
orig_texts, orig_labels = load_texts("/content/content/origs_NER", 0)   # оригинал
trans_texts, trans_labels = load_texts("/content/content/trans_NER", 1) # перевод

texts = orig_texts + trans_texts
labels = orig_labels + trans_labels

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df.head()

,text,label
0,Проиграный спор — читать фанфик по фэндому «Ро...,0
1,Мохнатое сердце чародея — читать фанфик по фэн...,0
2,"Попаданцы в мир ""Гарри Поттера"" — читать фанфи...",0
3,Проклятье — читать фанфик по фэндому «Роулинг ...,0
4,"То, что имеет значение — читать фанфик по фэнд...",0


In [17]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [18]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [21]:
def tokenize(texts):
    encodings = {"input_ids": [], "attention_mask": []}

    for text in tqdm(texts, desc="Tokenizing"):
        encoded = tokenizer(
            text,
            truncation=True,
            max_length=512
        )

        encodings["input_ids"].append(encoded["input_ids"])
        encodings["attention_mask"].append(encoded["attention_mask"])

    return encodings

In [22]:
train_encodings = tokenize(train_texts)
test_encodings = tokenize(test_texts)

Tokenizing: 100%|██████████| 342/342 [01:15<00:00,  4.51it/s]


In [20]:
class FanficDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {}

        for key in self.encodings:
            item[key] = torch.tensor(self.encodings[key][idx])

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [23]:
train_dataset = FanficDataset(train_encodings, train_labels)
test_dataset = FanficDataset(test_encodings, test_labels)

In [24]:
train_dataset[0]

{'input_ids': tensor([    2,   311,  2710,  2125,   776, 16410,  8558,   920,     5,     1,
          4781, 25568,   332,  1336,  8243,   865,   705,   332,  4318, 12363,
           644,   105,   296, 17704,  5952,  1072, 21591,  1336,   105,   283,
         25706,  1499,   751,  3658,   117,   117,  3086,  3277, 11444,  1512,
            30, 24499,  7843,  1348,  7081, 23101,  1142,   769, 28771, 25478,
         19319, 11410,  1560, 17441,   775,    16,  1363,  2710,  2125,   776,
         12175, 15960,   322, 18770,    35,   303,  7687,  4144,   299, 12666,
          1044,     5,  3280,   991, 13329,    18,    18,    18,  1218, 16410,
          8558,   869, 17717,    18,    18,    18,    23,    33,    33,    33,
            33,    33,    33,    33,    33,    33,    33,    33,    33,    33,
            33,    33,    33,    33,    33,    33,    33,    33,    33,    33,
            33,    33,    33,    33,    33,    33,    33,    33,    33,    33,
            33,    33,    33,    33,   

In [25]:
len(train_dataset)

1368

In [26]:
len(test_dataset)

342

In [27]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider train

In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [29]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,  # ← было tokenizer
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.633122,0.652047,0.642643
2,No log,0.554000,0.739766,0.727829
3,0.579940,0.532546,0.748538,0.731250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=513, training_loss=0.5770310230887192, metrics={'train_runtime': 33.7753, 'train_samples_per_second': 121.509, 'train_steps_per_second': 15.189, 'total_flos': 30263745429504.0, 'train_loss': 0.5770310230887192, 'epoch': 3.0})

In [32]:
trainer.save_model("fanfic_classifier")
tokenizer.save_pretrained("fanfic_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fanfic_classifier/tokenizer_config.json', 'fanfic_classifier/tokenizer.json')

In [33]:
import shutil
shutil.make_archive("/content/fanfic_classifier", "zip", "/content/fanfic_classifier")

'/content/fanfic_classifier.zip'

In [34]:
from google.colab import files
files.download("/content/fanfic_classifier.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>